# Week 12 Lab — Linear Programming and Gallery Walk
## SCIE1500 Science and Quantitative Reasoning — Semester 2, 2026

**Lab format:** Groups work on the LP problem (50 min) → Gallery Walk (20 min) → Debrief (10 min)

---
**Today:** We solve a dietary optimisation problem using linear programming (LP). Groups post their graphical solutions; everyone walks around and critiques each other's work.

---
## 📋 Scientific Scenario

![Linear programming feasible region with constraint lines](../images/W12_linear_programming.svg)

A hospital dietitian needs to design a daily meal plan for patients that meets nutritional requirements at minimum cost. Two food types are available:

- **Food X** (plant-based): costs **$2.50/serve**, provides 18 g protein, 220 kcal, 3 mg iron, 0.5 kg CO₂-e
- **Food Y** (chicken): costs **$4.00/serve**, provides 30 g protein, 280 kcal, 1 mg iron, 2.5 kg CO₂-e

Daily requirements per patient:
- Protein: at least 350 g
- Calories: at least 3,500 kcal
- Iron: at least 25 mg
- Carbon footprint: at most 20 kg CO₂-e per day

**Decision variables:** $x$ = serves of Food X, $y$ = serves of Food Y

---
## 🎯 Group Task & Learning Objectives

| Part | Topic | Time |
|------|-------|------|
| A | Formulate the LP and identify constraints | ~15 min |
| B | Graph the feasible region | ~20 min |
| C | Find the optimal solution using corner-point theorem | ~15 min |

In [ ]:
# Run this cell first — loads libraries for today's lab
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# LP parameters
cost = np.array([2.5, 4.0])     # objective: minimise 2.5x + 4y

# Constraints (as functions of x for plotting y = f(x))
def protein_line(x):
    '18x + 30y = 350  →  y = (350 - 18x) / 30'
    return (350 - 18*x) / 30

def calorie_line(x):
    '220x + 280y = 3500  →  y = (3500 - 220x) / 280'
    return (3500 - 220*x) / 280

def iron_line(x):
    '3x + y = 25  →  y = 25 - 3x'
    return 25 - 3*x

def carbon_line(x):
    '0.5x + 2.5y = 20  →  y = (20 - 0.5x) / 2.5'
    return (20 - 0.5*x) / 2.5

x_vals = np.linspace(0, 20, 300)

print("Constraint summary:")
print("  Protein:  18x + 30y ≥ 350")
print("  Calories: 220x + 280y ≥ 3500")
print("  Iron:     3x + y ≥ 25")
print("  Carbon:   0.5x + 2.5y ≤ 20")
print("  x ≥ 0, y ≥ 0")

---
## Part A: LP Formulation (~15 min)

**Objective function:**
$$\text{Minimise } Z = 2.5x + 4y$$

**Subject to:**
$$18x + 30y \geq 350 \quad \text{(protein)}$$
$$220x + 280y \geq 3500 \quad \text{(calories)}$$
$$3x + y \geq 25 \quad \text{(iron)}$$
$$0.5x + 2.5y \leq 20 \quad \text{(carbon)}$$
$$x \geq 0,\; y \geq 0$$

✏️ **A.1** For each constraint, identify which is a minimum requirement (≥) vs a maximum limit (≤). Why does this distinction matter for which side of the line is feasible?

```
ANSWERS:
...
```

In [ ]:
# B.1 — Plot all constraint lines
fig, ax = plt.subplots(figsize=(9, 8))

ax.plot(x_vals, protein_line(x_vals), "steelblue", lw=2, label="Protein: 18x+30y=350")
ax.plot(x_vals, calorie_line(x_vals), "firebrick", lw=2, label="Calories: 220x+280y=3500")
ax.plot(x_vals, iron_line(x_vals),    "seagreen",  lw=2, label="Iron: 3x+y=25")
ax.plot(x_vals, carbon_line(x_vals),  "purple",    lw=2, label="Carbon: 0.5x+2.5y=20")

# Shade feasible region (approximate)
y_grid = np.linspace(0, 20, 400)
xx, yy = np.meshgrid(x_vals, y_grid)
feasible = (
    (18*xx + 30*yy >= 350) &
    (220*xx + 280*yy >= 3500) &
    (3*xx + yy >= 25) &
    (0.5*xx + 2.5*yy <= 20) &
    (xx >= 0) & (yy >= 0)
)
ax.contourf(xx, yy, feasible.astype(float), levels=[0.5, 1.5], colors=["lightblue"], alpha=0.4)

ax.set_xlim(0, 18); ax.set_ylim(0, 18)
ax.set_xlabel("x (serves of Food X)"); ax.set_ylabel("y (serves of Food Y)")
ax.set_title("Hospital Diet LP: Feasible Region")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## Part C: Corner-Point Theorem (~15 min)

The minimum (or maximum) of a linear objective occurs at a **corner point** (vertex) of the feasible region.

**C.1** Find the corner points by solving pairs of constraints simultaneously.

Key intersections to check:
- Protein ∩ Iron: $18x + 30y = 350$ and $3x + y = 25$
- Protein ∩ Carbon: $18x + 30y = 350$ and $0.5x + 2.5y = 20$
- Calories ∩ Carbon: $220x + 280y = 3500$ and $0.5x + 2.5y = 20$
- Iron ∩ Carbon: $3x + y = 25$ and $0.5x + 2.5y = 20$

In [ ]:
# C.1 — Find corner points by solving pairs of constraints
from numpy.linalg import solve as linsolve

def find_corner(A_mat, b_vec):
    'Solve 2×2 linear system A x = b.'
    try:
        sol = linsolve(np.array(A_mat, dtype=float), np.array(b_vec, dtype=float))
        return sol if all(s >= -1e-6 for s in sol) else None
    except np.linalg.LinAlgError:
        return None

# Constraint matrix rows: [coeff_x, coeff_y]
corners_raw = [
    find_corner([[18, 30], [3, 1]],      [350, 25]),      # protein ∩ iron
    find_corner([[18, 30], [0.5, 2.5]],  [350, 20]),      # protein ∩ carbon
    find_corner([[220, 280], [0.5, 2.5]],[3500, 20]),     # calories ∩ carbon
    find_corner([[3, 1], [0.5, 2.5]],    [25, 20]),       # iron ∩ carbon
    find_corner([[220, 280], [3, 1]],    [3500, 25]),     # calories ∩ iron
]

corners = [c for c in corners_raw if c is not None]

print(f"{'Corner (x,y)':25s} {'Z=2.5x+4y':>12} {'Feasible?':>12}")
print("-" * 52)
for c in corners:
    x, y = c
    Z = 2.5*x + 4*y
    # Check all constraints
    ok = (18*x + 30*y >= 349.9 and 220*x + 280*y >= 3499.9 and
          3*x + y >= 24.9 and 0.5*x + 2.5*y <= 20.1 and x >= -0.01 and y >= -0.01)
    print(f"({x:6.2f}, {y:6.2f})          Z = {Z:8.2f}     {'✓' if ok else '✗'}")

Identify optimal corner.

In [ ]:
# C.2 — Identify optimal corner
feasible_corners = []
for c in corners:
    x, y = c
    ok = (18*x + 30*y >= 349.9 and 220*x + 280*y >= 3499.9 and
          3*x + y >= 24.9 and 0.5*x + 2.5*y <= 20.1 and x >= -0.01 and y >= -0.01)
    if ok:
        feasible_corners.append((x, y, 2.5*x + 4*y))

if feasible_corners:
    best = min(feasible_corners, key=lambda c: c[2])
    print(f"Optimal solution: x* = {best[0]:.2f}, y* = {best[1]:.2f}")
    print(f"Minimum cost: Z* = ${best[2]:.2f} per patient per day")
else:
    print("No feasible corners found — check constraint formulation")

Scipy LP verification.

In [ ]:
# C.3 — Scipy LP verification
from scipy.optimize import linprog

# Minimise 2.5x + 4y
# Constraints: convert all to ≤ form (negate ≥ constraints)
c_obj = [2.5, 4.0]

A_ub = [
    [-18, -30],    # -protein ≤ -350
    [-220, -280],  # -calories ≤ -3500
    [-3, -1],      # -iron ≤ -25
    [0.5, 2.5],    # carbon ≤ 20
]
b_ub = [-350, -3500, -25, 20]

result = linprog(c_obj, A_ub=A_ub, b_ub=b_ub, bounds=[(0,None),(0,None)], method='highs')
print(f"scipy linprog result:")
print(f"  x* = {result.x[0]:.4f} serves of Food X")
print(f"  y* = {result.x[1]:.4f} serves of Food Y")
print(f"  Z* = ${result.fun:.4f} per patient per day")


---
## ✅ Submission Exercise — Batch 4 (due Tuesday 27 October, 11:59 pm)

**Submit via LMS. Covers Weeks 10–12 (hypothesis testing, trigonometry, linear programming).**

Work individually. Show all working. Write Python code where indicated.
Upload your **completed notebook (.ipynb)** to the LMS — do not submit screenshots.
        

In [ ]:

# Fill in your details before submitting
student_name   = "Your full name"
student_number = "Your student number (e.g. 23456789)"
print(f"Submission by: {student_name}  |  Student #{student_number}")
        


---
### Q1 — Hypothesis Testing (Week 10)

A water authority claims average daily water consumption in a suburb is **no more than 180 L per person**. A sample of 36 households gives $\bar{x} = 191$ L and $s = 42$ L.

**(a)** State $H_0$ and $H_1$ for a one-tailed test.
        


**Answer (a):**

...write your answer here...
        


**(b)** Calculate the $t$-statistic. Show all working.
        


**Answer (b):**

...write your working here...
        


**(c)** At $\alpha = 0.05$ ($df = 35$, one-tailed critical value $\approx 1.690$), state your conclusion.
        


**Answer (c):**

...write your conclusion here...
        


**(d)** Compute the exact $p$-value using `scipy.stats.t`.
        

In [ ]:

# Q1d — exact p-value
from scipy import stats

# Your code here

        


---
### Q2 — Trigonometric Modelling (Week 11)

Daylight hours in a southern Australian city vary seasonally. The longest day has 14.5 hours (December 22) and the shortest has 9.5 hours (June 21).

**(a)** Find the amplitude $A$ and midline $D$.
        


**Answer (a):**

...write your working here...
        


**(b)** The period is 365 days. Find $B = 2\pi/365$.
        


**Answer (b):**

...write your answer here...
        


**(c)** Write the function $f(t)$ where $t$ = day of year, with $f(1) \approx 14.5$ (January 1 is near the longest day). Use an appropriate phase shift.
        


**Answer (c):**

...write your function here...
        


**(d)** Plot $f(t)$ for one year and calculate the total daylight hours over the year using `numpy.trapezoid`.
        

In [ ]:

# Q2d — plot and integrate
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Your code here

        


---
### Q3 — Linear Programming (Week 12)

A farmer grows wheat ($x$ ha) and canola ($y$ ha) on 200 ha. Constraints:
- Land: $x + y \leq 200$
- Water: $3x + 5y \leq 800$ (ML)
- Labour: $x + 2y \leq 300$ (days)
- Non-negativity: $x \geq 0$, $y \geq 0$

Profit: wheat $180/ha, canola $250/ha. **Maximise** $Z = 180x + 250y$.

**(a)** Graph the feasible region. Label each constraint line and shade the feasible region.
        


**Answer (a):** *(sketch or description)*

...write your description or attach a sketch here...
        


**(b)** Identify all corner points (intersections of constraint boundaries within the feasible region).
        


**Answer (b):**

...list corner points and show working here...
        


**(c)** Evaluate $Z$ at each corner point. State the optimal solution.
        


**Answer (c):**

...write your working and conclusion here...
        


**(d)** Verify your answer using `scipy.optimize.linprog`. (Note: `linprog` minimises — negate the objective to maximise.)
        

In [ ]:

# Q3d — linprog verification
from scipy.optimize import linprog

# Your code here

        